# From raw pyControl data to behavioural plots

This notebook is a guided first pass through the analysis pipeline. We will:

1. read a raw pyControl `.tsv` file and learn what each kind of row means;
2. inspect `import_data` in execution order and examine its output;
3. inspect `extract_trials` in execution order and examine its output;
4. independently recreate the ideas behind `plot_single_session.py` and `plot_rank_proportions.py`.

The plotting sections deliberately contain **no solution plotting code**. They give you data, specifications, aesthetic conventions, and checks. Try them before reading the repository's plotting modules.

## 0. Setup

Run this notebook from anywhere inside the repository. The next cell finds the project root by looking for `src` and `data`.

In [ ]:
from pathlib import Path
import inspect
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def find_project_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'src').is_dir() and (candidate / 'data').is_dir():
            return candidate
    raise FileNotFoundError('Could not find a parent containing both src/ and data/.')

ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.behavior_import.import_data import (
    collect_tsv_files, expand_content_columns, extract_session_towers,
    group_by_subj_and_ses, import_data,
)
from src.behavior_import.extract_trials import (
    ALIASES, extract_trials, standardize_variables, transpose_trials,
)
from src.behavior_analysis.get_believed_value import compute_believed_value
import src.behavior_visualization.plot_style  # applies the shared matplotlib defaults

SESSION_DIR = ROOT / 'data/3x3_field_blocked_reward_bandit/cohort-01/rawdata/sub-05_id-MY_72_L/ses-91_date-20260330'
RAW_ROOT = SESSION_DIR.parents[1]
files = sorted(SESSION_DIR.glob('*.tsv'))
print('Project root:', ROOT)
print('Files:', [f.name for f in files])

## A tiny coding vocabulary

You do not need to know Python before starting. These are the five ideas used throughout the notebook:

- A **variable** is a name attached to a value. In `mouse = 'MY_72_L'`, the name is `mouse` and the value is text.
- A **string** (`str`) is text, written inside quotes: `'A1'`.
- A **list** (`list`) is an ordered sequence, written with square brackets: `['A1', 'B1', 'C1']`. We select an item by position; Python starts counting at zero, so `towers[0]` is `'A1'`.
- A **dictionary** (`dict`) stores named values, written with curly braces: `{'choice': 'A1', 'reward': 4}`. The names are called **keys**. `trial['choice']` retrieves the value stored under the key `'choice'`.
- A pandas **DataFrame** is a rectangular table with named columns and numbered rows, much like a spreadsheet.

A **data type** tells us what kind of value something is and therefore what operations make sense. `type(value)` asks Python for its type. `len(value)` asks how many items it contains. A function is a named set of instructions: it receives input, performs operations, and may return output.

In [ ]:
mouse = 'MY_72_L'
towers = ['A1', 'B1', 'C1']
one_trial = {'choice': 'A1', 'reward': 4}
small_table = pd.DataFrame([one_trial, {'choice': 'B1', 'reward': 1}])

print(mouse, type(mouse))
print(towers, type(towers), 'first item:', towers[0])
print(one_trial, type(one_trial), "choice value:", one_trial['choice'])
print('DataFrame type:', type(small_table))
small_table

### How to read the code cells

- `=` means “store the value on the right under the name on the left.” It does not mean “is mathematically equal to.”
- `function_name(input)` calls a function. For example, `len(towers)` asks `len` to count the list.
- `object.method(...)` asks an object to perform one of its operations. For example, `raw.head(5)` asks a DataFrame to show its first five rows.
- Lines beginning with `#` are comments for people; Python ignores them.
- The value on the last line of a notebook cell is displayed automatically. `print(...)` displays a value deliberately.

Do not worry about memorising syntax. The goal is to recognise the shape of the data as it changes.

## 1. Anatomy of a raw pyControl file

A raw file is a tab-separated event log with four columns:

- `time`: seconds since this recording file began; it resets if recording restarts.
- `type`: broad row category (`info`, `state`, `event`, `variable`, and occasionally `error`).
- `subtype`: a more specific label. Blank subtypes are read by pandas as `NaN`.
- `content`: the payload. Its meaning depends on the row type; variable rows often contain JSON.

This directory has **two files for one session**. That is intentional and important: the recording was restarted, so downstream code must merge them.

In [ ]:
raw = pd.read_csv(files[0], sep='\t')
print('shape:', raw.shape)
raw.head(14)

### What is printed?

- `info`: recording metadata printed once at the beginning/end (task, setup, subject, timestamps, software versions).
- `state`: entry into a task state, such as waiting for initiation or choice.
- `event`: hardware inputs, timers, and sync pulses. These are the fine-grained event stream.
- `variable / run_start`: one JSON snapshot of task variables at startup. The importer uses this to identify choice and initiation towers.
- `variable / print`: one JSON trial summary. This is the main input to `extract_trials`.
- `variable / run_end`: final variable snapshot when present.
- `error`: a task/runtime message worth inspecting; its presence does not automatically invalidate every trial.

A useful distinction: **rows are not trials**. Many state/event rows happen within one trial; a `variable / print` row summarizes that trial.

In [ ]:
for path in files:
    frame = pd.read_csv(path, sep='\t')
    print(f'\n{path.name}: {len(frame)} rows')
    display(frame.groupby(['type', 'subtype'], dropna=False).size().rename('n_rows').to_frame())

### Read the JSON payloads

The text in `content` is not yet a Python dictionary. `json.loads` parses it. Compare the startup snapshot with the first trial summary. Before running the cell, predict which keys will occur in both.

In [ ]:
run_start_text = raw.loc[(raw['type'] == 'variable') & (raw['subtype'] == 'run_start'), 'content'].iloc[0]
trial_text = raw.loc[(raw['type'] == 'variable') & (raw['subtype'] == 'print'), 'content'].iloc[0]
run_start_dict = json.loads(run_start_text)
first_trial_dict = json.loads(trial_text)

print('run_start keys:')
print(sorted(run_start_dict))
print('\nfirst printed trial:')
first_trial_dict

#### Check your understanding

1. Why can `time` not be used directly as a continuous session clock after two files are merged?
2. Which rows would you use to reconstruct every poke? Which rows would you use for one observation per completed trial?
3. Find the `error` row in the first file and inspect nearby rows. What happened, and why might retaining the second file matter?
4. Compare the last printed trial of file 1 with the first printed trial of file 2. Which counters reset?

In [ ]:
error_rows = raw.index[raw['type'].eq('error')].tolist()
for i in error_rows:
    display(raw.loc[max(0, i - 4): min(len(raw) - 1, i + 4)])

for path in files:
    frame = pd.read_csv(path, sep='\t')
    printed = frame.loc[(frame['type'] == 'variable') & (frame['subtype'] == 'print'), 'content'].map(json.loads)
    print(path.name, 'first num_t =', printed.iloc[0].get('num_t'), 'last num_t =', printed.iloc[-1].get('num_t'))

## 2. `import_data`: from files to subjects and sessions

First read the function exactly as Python sees it. The numbers are display-only, so they remain accurate if the source file changes.

In [ ]:
def show_source(function):
    lines, start = inspect.getsourcelines(function)
    for number, line in enumerate(lines, start):
        print(f'{number:4}: {line}', end='')

show_source(import_data)

### What transformation does `import_data` perform?

**Input:** the path to a folder. A path is simply text describing where a file or folder lives.

**Output:** one nested dictionary containing all subjects and sessions found below that folder. “Nested” means that dictionaries contain other dictionaries.

```text
data dictionary
└── subject ID key (string, for example 'MY_72_L')
    └── session key (string, for example 'ses-91_date-20260330')
        ├── 'data'              → expanded DataFrame, or list after a restart
        ├── 'df'                → original four-column DataFrame, or list after a restart
        ├── 'problem'           → integer problem number
        ├── 'choice_towers'     → set of tower-name strings
        └── 'initiation_tower'  → tower-name string, or None
```

The outer key answers **which mouse?** The next key answers **which session?** The innermost keys answer **what do we know about that session?**

Now read the function one operation at a time:

1. The signature accepts a string or `Path`; the docstring promises a nested dictionary.
2. `Path(root).resolve()` makes the input absolute.
3. `collect_tsv_files` searches inside the folder and all folders below it. Every `.tsv` becomes a DataFrame. Its temporary output is a list of pairs: `(file path, DataFrame)`.
4. `group_by_subj_and_ses` reads subject/session names from each file's folder path. It changes the list into a dictionary whose keys are `(subject, session)` pairs. Each value is a list of DataFrames, ordered by recording time.
5. `subjects = {}` creates the output container; the set comprehension discovers subject IDs.
6. For each subject, relevant sessions are selected and sorted by date/session number.
7. `num_problem` begins at 1. The first session's tower set becomes the comparison baseline.
8. The code intends to compare each later session's tower set with the previous one and increment the problem number. However, `towers` and `prev_towers` are dictionaries: `set(towers)` compares their keys (`choice_towers`, `initiation_tower`), not the tower values. As written, this condition normally never detects a switch. A correct comparison would explicitly compare `towers['choice_towers']` (and decide separately whether initiation tower matters).
9. The code makes two independent copies of every DataFrame. `original_dfs` remains unchanged and is stored under `df`. It has exactly the four TSV columns: `time`, `type`, `subtype`, and `content`.
10. `expand_content_columns` acts on the second copy. It turns JSON keys inside `content` into new DataFrame columns. For example, the JSON key `num_t` becomes a column named `num_t`. This expanded copy is stored under `data`.
11. One-file sessions store a DataFrame; restarted sessions store a list of DataFrames. This is a deliberate union type to notice.
12. Session metadata (`problem`, choice towers, initiation tower) is attached, a summary is printed, and the nested dictionary is returned.

Why copy twice? `expand_content_columns` changes the DataFrame it receives. Passing it a copy ensures that expansion cannot accidentally add columns to `df`. Downstream, `extract_trials` deliberately reads `data`, because it needs the expanded representation. `df` remains available when we want to inspect the file exactly as it was read from disk.

### Open the helper functions

These helpers contain the mechanics hidden behind the three high-level calls. Read them in this order: collect → group → tower metadata → expand JSON.

In [ ]:
for fn in [collect_tsv_files, group_by_subj_and_ses, extract_session_towers, expand_content_columns]:
    print('\n' + '=' * 24, fn.__name__, '=' * 24)
    show_source(fn)

### Run the stages separately

This is the same pipeline as `import_data`, paused after each transformation.

In [ ]:
collected = collect_tsv_files(RAW_ROOT)
grouped = group_by_subj_and_ses(collected)
sample_group = grouped[('MY_72_L', 'ses-91_date-20260330')]

print('files collected:', len(collected))
print('group keys:', list(grouped))
print('recordings in sample session:', len(sample_group))
print('tower metadata:', extract_session_towers(sample_group[0]))
print('columns before expansion:', list(sample_group[0].columns))
expanded_example = expand_content_columns(sample_group[0].copy())
print('number of columns after expansion:', len(expanded_example.columns))
magnitude_columns = [c for c in expanded_example.columns if c.startswith('curr_rew_mag.')]
expanded_example.loc[expanded_example['subtype'].eq('print'), ['time', 'num_t', 'chosen_tow', *magnitude_columns]].head()

In [ ]:
data = import_data(RAW_ROOT)
session = data['MY_72_L']['ses-91_date-20260330']

print('subjects:', list(data))
print('sessions:', list(data['MY_72_L']))
print('session keys:', list(session))
print('data type:', type(session['data']).__name__, 'with', len(session['data']), 'recordings')
print('df type:', type(session['df']).__name__, 'with', len(session['df']), 'recordings')
print('raw df columns:', list(session['df'][0].columns))
print('expanded data column count:', len(session['data'][0].columns))
print('data and df are the same object:', session['data'][0] is session['df'][0])
print('problem:', session['problem'])
print('choice towers:', session['choice_towers'])
print('initiation tower:', session['initiation_tower'])

### The `import_data` output, key by key

The cell below builds a data dictionary: a small table describing every innermost key. `NoneType` means “no value is available.” A `set` resembles a list but has no meaningful order and cannot contain duplicates.

Both `data` and `df` have a type that depends on the session. Each is one DataFrame when there is one recording file, but a list of DataFrames when recording restarted. Here both are lists of length two.

The difference is their columns: `df` is the original four-column event log, while `data` contains those same four columns plus the JSON-derived columns. They are separate objects, so changing one does not change the other. `import_data` has organised and expanded the recordings, but it has not yet made one row per trial.

In [ ]:
def describe_value(value):
    """Return beginner-friendly information about one Python value."""
    if isinstance(value, pd.DataFrame):
        return 'DataFrame', f'{value.shape[0]} rows × {value.shape[1]} columns', list(value.columns[:5])
    if isinstance(value, list):
        item_types = sorted({type(item).__name__ for item in value})
        if value and all(isinstance(item, pd.DataFrame) for item in value):
            example = [f'{item.shape[0]} rows × {item.shape[1]} columns' for item in value]
        else:
            example = value[:2]
        return 'list', f'{len(value)} items; item type(s): {item_types}', example
    if isinstance(value, set):
        return 'set', f'{len(value)} unique items', sorted(value)
    return type(value).__name__, 'one value', value

import_rows = []
for key, value in session.items():
    value_type, size, example = describe_value(value)
    import_rows.append({'key': key, 'data type': value_type, 'size / contents': size, 'example': example})

pd.DataFrame(import_rows)

### Follow one piece of information through the transformation

Take the chosen tower on the first trial:

1. **On disk:** it is characters inside the `content` cell of a `.tsv` row: `... \"chosen_tow\": \"C1\" ...`.
2. **After `pd.read_csv`:** that entire JSON object is still one string in the DataFrame's `content` column. A copy of this four-column table is retained under the session key `df`.
3. **In the expanded copy:** `json.loads` converts the string to a dictionary, and `pd.json_normalize` creates a `chosen_tow` column. The value in the first printed-trial row is now the Python string `'C1'`.
4. **After `import_data`:** the raw table is reachable through `data['MY_72_L']['ses-91_date-20260330']['df'][0]`, while the expanded table is reached through `data['MY_72_L']['ses-91_date-20260330']['data'][0]`. The choice is not yet stored as a clean 93-item list; that is the job of `extract_trials`.

#### Mini-exercises

1. Draw the nesting of `data` on paper and label which keys identify subject, session, and session properties.
2. Write one expression that returns the first recording DataFrame without hard-coding `[0]` for ordinary one-file sessions.
3. Explain the problem-detection bug described above. Why is comparing `towers['choice_towers']` as a **set** more appropriate than comparing lists here? When might set comparison still hide a meaningful change?
4. Confirm that `session['data'][0] is session['df'][0]` is `False`. Compare their column names, then change a value in a temporary copy of one and confirm that the other is unaffected.

## 3. `extract_trials`: from an event log to one value per trial

`import_data` gave us full event-log DataFrames. `extract_trials` keeps only the printed trial summaries and adds analysis-ready values to each session dictionary. It changes the same `data` object and also returns it. Changing an existing object is called **mutating it in place**.

The central transformation is:

```text
many event-log rows
→ keep only variable / print rows
→ parse each JSON string into one dictionary per trial
→ turn the list of trial dictionaries sideways
→ one session dictionary containing parallel lists
```

“Parallel lists” means that position 0 always describes trial 1, position 1 always describes trial 2, and so on. For example, `session['choice'][9]`, `session['blocks'][9]`, and `session['reward_magnitudes'][9]` all describe the same tenth trial.

In [ ]:
show_source(extract_trials)

### What transformation does `extract_trials` perform?

1. `safe_json_load` converts JSON strings to Python objects and safely maps missing/malformed content to `{}`.
2. The two outer loops visit every subject, then every session.
3. A one-file session initially contains one DataFrame, whereas this restarted session contains a list of two. A lone DataFrame is wrapped inside a one-item list so the remaining code can always loop over “a list of recordings.”
4. Within each recording DataFrame, the code selects only rows where `type == 'variable'` and `subtype == 'print'`, then sorts those rows by time. Hundreds of state/event rows are discarded from this trial-level representation (they remain available in `data`/`df`).
5. Each selected `content` value begins as JSON text. `safe_json_load` turns it into a Python dictionary. Port-finding dictionaries containing `num_t_found` are removed because they do not represent completed task trials.
6. `transpose_trials` applies `ALIASES`: different historical spellings become one stable key. It also turns data “sideways”: `[{'num_t': 1}, {'num_t': 2}]` becomes `{'trial': [1, 2]}`.
7. If no trials exist, empty debug fields are attached and that session is skipped.
8. The flattened raw dictionaries are searched for good/bad reversal keys. This distinguishes task variants that never printed those counters.
9. `trial_variables` and `trial_info` retain intermediate representations for debugging.
10. For a one-file session, every newly created per-variable list is added directly to the session dictionary. Thus `session['choice']` becomes a list of chosen-tower strings, not a DataFrame column.
11. For a restarted session, lists from the two recordings must be joined. Different keys need different rules. Here `trial` changes from `[1, ..., 31]` plus `[1, ..., 62]` to `[1, ..., 93]`. Cumulative block/count values are offset rather than simply repeated. Ordinary observations such as choice strings are appended.
12. Post-processing creates convenient derived representations. A list of reward dictionaries becomes a dictionary of tower lists; choice strings become 0/1 indicators by tower; chosen ranks become 0/1 indicators by rank; missing rank labels may be inferred from changes in cumulative rank counts.
13. Reversal keys absent from the original file are removed rather than leaving misleading lists of `None`.
14. The mutated input dictionary is returned.

### Alias normalisation and transposition

Older/newer task versions print different spellings. `ALIASES` is a compatibility layer: for example, both `num_trials` and `num_t` become `trial`; `chosen_tower` and `chosen_tow` become `choice`. Missing variables become `None`, preserving aligned list lengths.

In [ ]:
show_source(standardize_variables)
print()
show_source(transpose_trials)

toy_trials = [
    {'num_t': 1, 'chosen_tow': 'A1'},
    {'num_trials': 2, 'chosen_tower': 'B1'},
]
print('\nstandardized first trial:', standardize_variables(toy_trials[0], ALIASES))
print('transposed subset:', transpose_trials(toy_trials, ALIASES, keep=['trial', 'choice', 'chosen_rank']))

### Run it and inspect the contract

Before running: predict the length and final value of `trial`, and whether `good_reversals` will be present for this performance-independent recording.

In [ ]:
returned = extract_trials(data)
assert returned is data  # mutation plus return
session = data['MY_72_L']['ses-91_date-20260330']

print('number of trials:', len(session['trial']))
print('trial boundary:', session['trial'][:3], '...', session['trial'][-3:])
print('number of blocks:', session['blocks'][-1])
print('has_good / has_bad:', session['has_good'], session['has_bad'])
print('good_reversals key present:', 'good_reversals' in session)
print('derived keys:', [k for k in session if k.startswith(('reward_', 'choices_'))])

In [ ]:
trial_table = pd.DataFrame({
    'trial': session['trial'],
    'block': session['blocks'],
    'trial_in_block': session['trials_in_block'],
    'choice': session['choice'],
    'chosen_rank': session['chosen_rank'],
    'chosen_magnitude': session['chosen_magnitude'],
})
trial_table.head(10)

### The final session structure

`extract_trials` does not replace the output of `import_data`. It **adds keys** to the same innermost session dictionary. The outer subject/session nesting remains unchanged.

The added keys fall into four groups:

- **One value per trial:** lists such as `trial`, `choice`, `blocks`, `chosen_rank`, and `reward_magnitudes`. Each list has 93 items here. An item may itself be a number, string, dictionary, list, Boolean, or `None`.
- **Derived dictionaries of parallel lists:** `reward_magnitudes_by_tower`, `choices_by_tower`, `reward_magnitude_by_arm`, `reward_magnitude_by_rank`, and `choices_by_rank`. Their inner keys name towers/ranks; each inner value is a 93-item list.
- **Session-level values:** `has_good` and `has_bad` are single Boolean (`bool`) values, not per-trial lists.
- **Debug/intermediate values:** `trial_info` and `trial_variables` preserve one entry per recording file. Because this session has two files, their outer lists have length two.

The table below reports **every key currently present**, its Python type, its size, the type of values inside it, and a short preview. This is a practical data dictionary generated from the real output.

In [ ]:
def short_preview(value):
    if isinstance(value, pd.DataFrame):
        return f'columns: {list(value.columns[:4])} ...'
    if isinstance(value, list):
        if value and all(isinstance(item, pd.DataFrame) for item in value):
            return [f'DataFrame{item.shape}' for item in value]
        return repr(value[:3])[:120]
    if isinstance(value, dict):
        return f'inner keys: {list(value)[:6]}'
    if isinstance(value, set):
        return repr(sorted(value))
    return repr(value)[:120]

def inner_type(value):
    if isinstance(value, list):
        return ', '.join(sorted({type(item).__name__ for item in value})) if value else '(empty)'
    if isinstance(value, dict):
        return ', '.join(sorted({type(item).__name__ for item in value.values()})) if value else '(empty)'
    if isinstance(value, pd.DataFrame):
        return 'columns contain mixed types'
    return '—'

def value_size(value):
    if isinstance(value, pd.DataFrame):
        return f'{value.shape[0]} rows × {value.shape[1]} columns'
    if isinstance(value, (list, dict, set)):
        return len(value)
    return 1

session_dictionary = pd.DataFrame([
    {
        'key': key,
        'outer type': type(value).__name__,
        'size': value_size(value),
        'type(s) inside': inner_type(value),
        'preview': short_preview(value),
    }
    for key, value in sorted(session.items())
])
pd.set_option('display.max_rows', None)
display(session_dictionary)
pd.reset_option('display.max_rows')

### Important keys in plain language

| Key | Type | Meaning |
|---|---|---|
| `trial` | list of integers | Continuous trial numbers after recordings are merged. |
| `blocks` | list of integers | Cumulative block number for each trial. |
| `trials_in_block` | list of integers | Position of each trial within its current block. |
| `choice` | list of strings | Tower chosen on each trial, such as `'C1'`. |
| `chosen_rank` | list of strings | Whether that choice was currently best, second, or third. |
| `reward_magnitudes` | list of dictionaries | On each trial, maps every tower to its available magnitude. |
| `reward_magnitudes_by_tower` | dictionary of lists | The same magnitudes reorganised so each tower has one list across trials. |
| `choices_by_tower` | dictionary of integer lists | One-hot representation: 1 for the chosen tower, 0 for the others, on every trial. |
| `rank_counts` | list of dictionaries | Running cumulative counts of choices by rank after each trial. |
| `choices_by_rank` | dictionary of integer lists | One-hot representation of best/second/third choices. |
| `ema_best_arm_choices` | list of numbers | Smoothed recent tendency to choose the best arm; may contain missing values in some task versions. |
| `has_good`, `has_bad` | Boolean | Whether this session actually printed good/bad reversal variables. |
| `trial_info` | list of lists of dictionaries | Parsed trial JSON, kept separately for each recording file. |
| `trial_variables` | list of dictionaries of lists | Standardised/transposed variables before the two files are merged. |

Other keys in the generated table follow the same rule: use the outer type, inner type, size, and preview together to understand their shape.

### Why are some extracted values `None`? Task versions share one extractor

`extract_trials` is designed to read several related versions of the task. The blocked/performance-dependent reversal versions print variables that this drifting-value recording does not use. The `ALIASES` dictionary defines one common set of output names for all versions. During transposition, if none of a key's possible spellings occurs in a trial dictionary, the extractor inserts `None` for that trial so all parallel lists remain the same length.

This means a field such as `ema_best_arm_choices` is still a **list**, but every item inside that list has type `NoneType`. `None` means “no value was printed for this field on this trial.” Here it usually means **not applicable to this task version**, rather than a corrupted or failed trial.

For this drifting session, these 93-item lists are entirely `None`:

| Key | Mainly used for | Why it is not populated here |
|---|---|---|
| `best_arm` | Tasks that explicitly print a discrete best arm | The drifting task prints changing reward magnitudes but does not print this field. |
| `ema_best_arm_choices` | Performance-dependent blocked/reversal tasks | No best-choice EMA is logged by this protocol. |
| `ema_second_arm_choices` | Performance-dependent blocked/reversal tasks | No second-choice EMA is logged by this protocol. |
| `can_switch` | Rules that permit a block/reversal after a performance criterion | This drifting protocol does not use that switching criterion. |
| `num_trials_until_switch` | Performance-dependent switching logic | The associated switching counter is not part of this recording. |
| `total_num_trials_until_switch` | Performance-dependent switching logic | The associated cumulative counter is not part of this recording. |
| `trials_since_last_reversal` | Explicit reversal-based task variants | This field is not printed by the drifting protocol. |

Good/bad reversal counters are handled slightly differently. After extraction, `has_good` and `has_bad` are `False`, and the all-`None` `good_reversals`/`bad_reversals` keys are removed entirely. That prevents downstream analysis from mistaking an unavailable counter for a real counter.

A useful rule: first ask whether a variable belongs to the protocol you ran. Only treat absent values as a data-quality problem if that protocol was expected to print them.

In [ ]:
# Find per-trial lists that exist but contain no recorded values.
all_none_keys = [
    key for key, values in session.items()
    if isinstance(values, list)
    and len(values) > 0
    and all(value is None for value in values)
]
print('Lists containing only None:', all_none_keys)
print('has_good:', session['has_good'], 'has_bad:', session['has_bad'])
print('good_reversals present:', 'good_reversals' in session)
print('bad_reversals present:', 'bad_reversals' in session)

### Follow the same choice all the way to the final output

For trial 1, the raw JSON contains `chosen_tow: C1`. The transformation is:

```text
raw content string
→ {'num_t': 1, ..., 'chosen_tow': 'C1', ...}       one parsed trial dictionary
→ {'trial': [1, ...], 'choice': ['C1', ...], ...}  standardised parallel lists
→ session['choice'][0] == 'C1'                     direct analysis value
→ session['choices_by_tower']['C1'][0] == 1        derived one-hot value
→ session['choices_by_tower']['A1'][0] == 0
→ session['choices_by_tower']['B1'][0] == 0
```

The information has not changed; it has been reorganised into forms that make later questions and plots easier.

In [ ]:
# Structural checks: these are invariants the extractor should satisfy.
n_trials = len(session['trial'])
assert session['trial'] == list(range(1, n_trials + 1))
assert all(len(values) == n_trials for values in session['reward_magnitudes_by_tower'].values())
assert all(len(values) == n_trials for values in session['choices_by_tower'].values())
assert all(sum(session['choices_by_tower'][tower][i] for tower in session['choices_by_tower']) == 1 for i in range(n_trials))
assert all(sum(session['choices_by_rank'][rank][i] for rank in session['choices_by_rank']) == 1 for i in range(n_trials))
print('All alignment and one-hot checks passed.')

#### Mini-exercises

1. Use `trial_info` to display the raw dictionary for trial 10, then compare it with row 10 of `trial_table`.
2. Explain why ordinary concatenation is wrong for `trial`, `blocks`, and cumulative `rank_counts`.
3. From `choices_by_tower`, recover the original `choice` for the first five trials.
4. Write a check that the chosen magnitude equals the reward magnitude at the chosen tower on every trial.
5. Find the trial indices at which `blocks` increments using `np.diff`. Be careful about zero-based array positions versus displayed trial numbers.

## 4. Plotting exercise A — recreate the single-session summary

Do **not** import or call `plot_single_session`. Build the figure from `session`. Your target contains:

1. a full-width reward-magnitude panel, one line per tower, with black dots at the magnitude actually chosen;
2. a bottom-left bar chart of choice counts by tower with percentages above bars;
3. a bottom-right bar chart of final choice counts by rank with percentages above bars;
4. dashed vertical lines wherever a cumulative block/reversal counter increments.

In [ ]:
# Exercise A workspace — write your plotting code below.
# Suggested decomposition: prepare arrays -> identify boundaries -> create layout -> draw each panel -> style.

magnitude_by_tower = session['reward_magnitudes_by_tower']
choice_by_tower = session['choices_by_tower']
rank_counts = session['rank_counts'][-1]
n_trials = len(session['trial'])

# TODO: create your figure here.
raise NotImplementedError('Exercise A: recreate the single-session figure')

### Self-check for exercise A

- Exactly one black choice marker appears per trial.
- Tower bar counts sum to `n_trials`; rank bar counts also sum to `n_trials`.
- Magnitude values are plotted against trial position and the three tower traces remain distinguishable.
- Block boundaries align with changes in `session['blocks']`, not with arbitrary time gaps.
- The plot still works if EMA keys are absent.
- You did not copy or call the repository plotting function.

## 5. Plotting exercise B — true-value and believed-value rank proportions

The word **best** can mean two different things. Keeping them separate is scientifically important.

### True value

The **true value** is the reward magnitude the task has actually assigned to each tower on the current trial. The experiment knows all three true values. `session['reward_magnitudes'][i]` contains that complete mapping for trial `i`, and `session['chosen_rank'][i]` says whether the chosen tower is truly best, second, or third on that trial.

### Believed value

The mouse cannot inspect the task's hidden mapping. In this analysis, its **believed value** for a tower is the reward magnitude it last observed when it previously chose that tower. The belief is therefore a model constructed by us, not a directly measured thought inside the mouse.

Believed rank is calculated using information available **before the current choice**:

1. Begin the session knowing no tower values.
2. Before trial `i`, look up the last observed reward magnitude for every tower that has already been chosen.
3. Rank only those previously seen towers from highest believed magnitude to lowest.
4. Record the believed rank of the tower chosen on trial `i`. If that tower has never been chosen before, its believed rank is `None` (unknown).
5. Only after recording the trial's pre-choice belief, update the chosen tower with its true magnitude on trial `i`. This prevents information from the current outcome leaking backward into the decision.

After a reversal, true and believed rank can disagree: the task's values change immediately, but the mouse's last-observed values remain stale until it samples the towers again.

### A small example

Suppose the true values are `A=4, B=2, C=1`, and the mouse has previously observed those values. If the task silently reverses to `A=1, B=2, C=4`, then C is now the **true best** tower. Before sampling again, however, the mouse's stored values still say A=4 and C=1, so A remains **believed best**. Choosing A on that trial is therefore a true-third choice but a believed-best choice.

These two classifications answer different questions:

- True-value proportions: *How often did behaviour select the objectively best available option?*
- Believed-value proportions: *How often did behaviour select the option that appeared best under this last-observation belief model?*

In [ ]:
believed_trials = compute_believed_value(
    choices=session['choice'],
    reward_magnitudes=session['reward_magnitudes'],
)

# Add the two rank definitions side by side for inspection.
rank_comparison = trial_table.copy()
rank_comparison = rank_comparison.rename(columns={'chosen_rank': 'true_chosen_rank'})
rank_comparison['believed_chosen_rank'] = [row['believed_chosen_rank'] for row in believed_trials]
rank_comparison['number_of_seen_towers'] = [row['n_seen'] for row in believed_trials]
rank_comparison['believed_top_two_tied'] = [row['tied'] for row in believed_trials]
rank_comparison['true_and_believed_agree'] = (
    rank_comparison['true_chosen_rank'] == rank_comparison['believed_chosen_rank']
)
rank_comparison.head(12)

Read the first rows carefully. Early believed ranks may be `None` because the chosen tower had no previously observed value. `number_of_seen_towers` shows how much of the choice set the belief model can currently rank.

When the top two believed magnitudes are equal, `believed_top_two_tied` is `True`. The repository function makes ordering reproducible by breaking equal-value ties using the tower name, but the tie flag remains available. In a full analysis, report how often ties occur and consider whether conclusions change if tied trials are excluded or treated fractionally.

In [ ]:
# Data-preparation exercise (not plotting): make TWO summaries per block.
true_block_rank_rows = []
believed_block_rank_rows = []
for block_number, rows in trial_table.groupby('block', sort=True):
    trial_positions = rows.index

    # TRUE VALUE TODO
    # Use rank_comparison.loc[trial_positions, 'true_chosen_rank'].
    # Count best/second/third and divide by ALL trials in this block.
    # Append a dictionary to true_block_rank_rows.
    # Required keys: block, total, best_prop, second_prop, third_prop

    # BELIEVED VALUE TODO
    # Use rank_comparison.loc[trial_positions, 'believed_chosen_rank'].
    # Exclude None/unknown ranks from best/second/third proportions.
    # The denominator is therefore the number of VALID believed-rank trials,
    # not necessarily every trial in the block. Also record how many were unknown.
    # Required keys: block, total, unknown, best_prop, second_prop, third_prop
    pass

true_rank_proportions = {'MY_72_L': true_block_rank_rows}
believed_rank_proportions = {'MY_72_L': believed_block_rank_rows}

print('True-value summary:')
display(true_rank_proportions)
print('Believed-value summary:')
display(believed_rank_proportions)

### The three target views — produced for both definitions

For each view below, create a true-value version and a believed-value version using identical category order, colours, and y limits. Matching scales are essential for honest visual comparison. Put **True Value** or **Believed Value** in every title; never leave the rank definition implicit.

1. **By mouse:** one panel per mouse; grey bars show mean ± SE across blocks, coloured connected points show individual blocks.
2. **By block:** one panel per block; grey bars show mean ± SE across mice, coloured connected points show individual mice.
3. **Average:** two panels; left summarizes across mice and overlays mouse trajectories, right summarizes across blocks and overlays block trajectories.
4. **Direct comparison:** add one compact figure in which true and believed mean proportions are adjacent or overlaid for Best, Second, and Third. Use colour/line style for the rank definition, while retaining the same mouse/block colour identities elsewhere. Also display the proportion of trials with unknown believed rank.

Statistical note: decide what the independent unit is before computing an SE. Across-mouse and across-block uncertainty answer different questions; the two-panel average view makes that distinction visible. True and believed summaries may also have different denominators because unknown believed ranks are excluded. Always show or report those denominators.

In [ ]:
# Exercise B workspace — no solution plotting code is provided.
# Recommended: write three functions with a shared styling helper.

def plot_rank_proportions_by_mouse(rank_data, value_definition):
    # value_definition should be the text 'True Value' or 'Believed Value'.
    # TODO
    raise NotImplementedError

def plot_rank_proportions_by_block(rank_data, value_definition):
    # TODO
    raise NotImplementedError

def plot_rank_proportions_average(rank_data, value_definition):
    # TODO
    raise NotImplementedError

def plot_true_vs_believed(true_rank_data, believed_rank_data):
    # TODO: make the direct comparison and disclose believed-rank coverage.
    raise NotImplementedError

# When implemented, call each of the first three functions once with
# true_rank_proportions and once with believed_rank_proportions.

### Self-check for exercise B

- Every true-value block's three proportions sum to 1 (allowing floating-point tolerance).
- Every believed-value block with at least one valid rank sums to 1 across best/second/third. Unknown trials are counted separately and excluded from that denominator.
- The true-value denominator equals all trials in a block; the believed-value denominator equals only trials whose chosen tower had a pre-choice believed rank.
- Every figure title states `True Value` or `Believed Value`.
- True and believed plots use identical axes and matching visual conventions, so differences cannot be caused by rescaling.
- The direct-comparison figure reports believed-rank coverage or the number/proportion unknown.
- Means and SEs are computed across the unit named in the panel title.
- All panels use the same category order and y-axis range.
- Subject and block colours remain stable across panels.
- Jitter changes x-position only, never the measured y-value.
- Empty subjects/blocks and one-observation SEs are handled without crashing.
- You did not copy or call `plot_rank_proportions.py`.

## 6. Reflection and extension

1. Which information is lost when the event stream is reduced to one dictionary per trial?
2. Which extractor outputs are raw measurements, and which are derived representations?
3. On which trials do true and believed chosen rank disagree? Are disagreements concentrated immediately after block changes? Why would that pattern be expected?
4. How does excluding unknown believed ranks change the population of trials being summarized? Could this make early behaviour look systematically different?
5. What assumptions does a line connecting Best → Second → Third imply visually? Is it useful here despite the x-axis being categorical?
6. Extend the workflow to several sessions and mice. Keep data preparation separate from plotting.
7. Only after completing your attempt, compare it with `src/behavior_visualization/plot_single_session.py` and `src/behavior_visualization/plot_rank_proportions.py`. Identify one design decision you would keep and one you would revise.